In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

In [ ]:
sns.set(style="whitegrid")

def load_and_inspect(file_path, name):
    """Loads a CSV and prints basic info."""
    print(f"\n--- Loading {name} Data ---")
    df = pd.read_csv(file_path)
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    return df

In [ ]:
file_sanitation = 'data/311_Service_Requests_-_Sanitation_Code_Complaints_-_Historical_20260203.csv'
file_vacant = 'data/311_Service_Requests_-_Vacant_and_Abandoned_Buildings_Reported_-_Historical_20260203.csv'

df_sanitation = load_and_inspect(file_sanitation, "Sanitation")
df_vacant = load_and_inspect(file_vacant, "Vacant Buildings")

In [ ]:
df_sanitation['Creation Date'] = pd.to_datetime(df_sanitation['Creation Date'], errors='coerce')
df_sanitation['Completion Date'] = pd.to_datetime(df_sanitation['Completion Date'], errors='coerce')

In [ ]:
df_vacant['Date Received'] = pd.to_datetime(df_vacant['DATE SERVICE REQUEST WAS RECEIVED'], errors='coerce')

In [ ]:
print("\n--- Missing Values (Percentage) ---")
print("Sanitation Top 5 Missing:")
print(df_sanitation.isnull().mean().sort_values(ascending=False).head(5))
print("\nVacant Buildings Top 5 Missing:")
print(df_vacant.isnull().mean().sort_values(ascending=False).head(5))

In [ ]:
plt.figure(figsize=(14, 6))
df_sanitation.set_index('Creation Date').resample('M').size().plot(label='Sanitation Complaints', alpha=0.7)
df_vacant.set_index('Date Received').resample('M').size().plot(label='Vacant Building Reports', color='orange', alpha=0.7)
plt.title('Monthly Volume of 311 Requests (2011 - 2019)')
plt.ylabel('Number of Requests')
plt.xlabel('Date')
plt.legend()
plt.tight_layout()
plt.savefig('eda_temporal_trends.png')
plt.show()

In [ ]:
top_wards_sanitation = df_sanitation['Ward'].value_counts().head(10)
top_wards_vacant = df_vacant['Ward'].value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(x=top_wards_sanitation.index, y=top_wards_sanitation.values, ax=axes[0], palette='Blues_d')
axes[0].set_title('Top 10 Wards: Sanitation Complaints')
axes[0].set_xlabel('Ward')
axes[0].set_ylabel('Count')

sns.barplot(x=top_wards_vacant.index, y=top_wards_vacant.values, ax=axes[1], palette='Oranges_d')
axes[1].set_title('Top 10 Wards: Vacant Building Reports')
axes[1].set_xlabel('Ward')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('eda_top_wards.png')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
top_nature = df_sanitation['What is the Nature of this Code Violation?'].value_counts().head(5)
sns.barplot(y=top_nature.index, x=top_nature.values, palette='viridis')
plt.title('Top 5 Types of Sanitation Violations')
plt.xlabel('Count')
plt.tight_layout()
plt.savefig('eda_sanitation_types.png')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
status_counts = df_vacant['IS THE BUILDING CURRENTLY VACANT OR OCCUPIED?'].value_counts()
status_counts.plot(kind='bar', color='firebrick', alpha=0.7)
plt.title('Reported Building Status (Vacant vs Occupied)')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('eda_vacant_status.png')
plt.show()

In [ ]:
df_sanitation['Resolution Days'] = (df_sanitation['Completion Date'] - df_sanitation['Creation Date']).dt.days

print("\n--- Sanitation Resolution Time Stats (Days) ---")
print(df_sanitation['Resolution Days'].describe())

plt.figure(figsize=(10, 5))
sns.histplot(df_sanitation['Resolution Days'].dropna(), bins=100, kde=False)
plt.title('Distribution of Resolution Time (Sanitation)')
plt.xlabel('Days to Complete')
plt.xlim(0, 60) # Zoom in on the first 2 months
plt.savefig('eda_resolution_time.png')
plt.show()

In [ ]:
ward_counts_s = df_sanitation['Ward'].value_counts()
top_10_s = ward_counts_s.head(10).sum()
rest_s = ward_counts_s.iloc[10:].sum()
total_s = top_10_s + rest_s

ward_counts_v = df_vacant['Ward'].value_counts()
top_10_v = ward_counts_v.head(10).sum()
rest_v = ward_counts_v.iloc[10:].sum()
total_v = top_10_v + rest_v

plot_df = pd.DataFrame({
    'Dataset': ['Sanitation', 'Vacant Buildings'],
    'Top 10 Wards': [top_10_s, top_10_v],
    'Rest of City': [rest_s, rest_v]
})

plot_df['Total'] = plot_df['Top 10 Wards'] + plot_df['Rest of City']
plot_df['Top 10 %'] = plot_df['Top 10 Wards'] / plot_df['Total']
plot_df['Rest %'] = plot_df['Rest of City'] / plot_df['Total']

fig, ax = plt.subplots(figsize=(10, 6))

color_top10 = ['lightblue', 'darkorange']
color_rest = ['blue', 'orange']

p1 = plt.bar(plot_df['Dataset'], plot_df['Top 10 %'], color=color_top10)
p2 = plt.bar(plot_df['Dataset'], plot_df['Rest %'], bottom=plot_df['Top 10 %'], color=color_rest, alpha=0.3)

for i, val in enumerate(plot_df['Top 10 %']):
    plt.text(i, val/2, f"{val:.1%}", ha='center', va='center', fontweight='bold', fontsize=12)
    
for i, val in enumerate(plot_df['Rest %']):
    plt.text(i, val/2 + plot_df['Top 10 %'][i], f"{val:.1%}", ha='center', va='center', fontsize=12)

legend_handles = [
    mpatches.Patch(color='lightblue', label='Sanitation: Top 10 Wards'),
    mpatches.Patch(color='blue', alpha=0.3, label='Sanitation: Rest of City'),
    mpatches.Patch(color='darkorange', label='Vacant: Top 10 Wards'),
    mpatches.Patch(color='orange', alpha=0.3, label='Vacant: Rest of City')
]

plt.legend(handles=legend_handles, bbox_to_anchor=(1.05, 1), loc='upper left', title="Concentration Breakdown")

plt.title('Share of Total Volume: Top 10 Wards vs Rest of City', fontsize=14)
plt.ylabel('Percentage of Total Reports')
plt.tight_layout()
plt.savefig('concentration_share_bar_fixed.png')
plt.show()

In [ ]:
common_wards = set(df_sanitation['Ward'].dropna().unique()).intersection(set(df_vacant['Ward'].dropna().unique()))
print(f"\nNumber of Wards present in both datasets: {len(common_wards)}")